# Construction of the pMEC9001, 2 and 3 vectors

The vector pMEC1049 vector was used in [Romaní et al. 2014](http://www.sciencedirect.com/science/article/pii/S096085241401757X) The pMEC1049 expresses a D-xylose metabolic pathway and has a hygromycin selectable marker. The details of the construction of pMEC1049 can be found [here](pMEC1049.ipynb).

This document describe the construction of the pMEC9001, 2 and 3 vectors. The pMEC9001 is the pMEC1049 with an additional expression cassette for the Saccharomyces cerevisiae gene HAA1(YPR008W). The pMEC9002 is the pMEC1049 with an additional expression cassette for S. cerevisiae PRS3(YHL011C) and pMEC9003 has both of them.

| Vector   | Relevant property                                            |
|----------|--------------------------------------------------------------|
| pMEC9001 | [HAA1](http://www.yeastgenome.org/locus/S000006212/overview) |
| pMEC9002 | [PRS3](http://www.yeastgenome.org/locus/S000001003/overview) |
| pMEC9003 | HAA1 & PRS3                                                  |

Normally, this would be done following the yeast pathway kit strategy by adding genes with a set of new promoters and terminators, but in this case a requirement was to retain the native promoters and terminators for HAA1 and PRS3.

The strategy involves linearizing the vector in two locations (before and after the xylose pathway) and adding the HAA1 and PRS3 expression cassettes amplified using tailed primers.


## The PRS3 construct

The PRS3 cassette was previously cloned according to the description below in vector YEpJCP according to this description:

"obtained by PCR amplification of fragment carrying PRS3 from Saccharomyces cerevisiae CEN.PK113-7D genomic DNA using appropriate primers and insertion into plasmid pGEM-T Easy. Cloning into YEplac195KanMX using EcoRI digestion sites." [Cunha et al. 2015](http://www.sciencedirect.com/science/article/pii/S0960852415006707)

Primers:

    P1: TTATCTTCATCACCGCCATAC
    P2: ACAAGAGAAACTTTTGGGTAAAATG

The exact same PRS3 fragment will be cloned in pMEC9002.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
from pydna.parsers import parse_primers

In [ ]:
p1,p2 = parse_primers('''
>P1
TTATCTTCATCACCGCCATAC
>P2
ACAAGAGAAACTTTTGGGTAAAATG
''')

In [ ]:
from pydna.readers import read

The S288C_YHL011C_PRS3_flanking.fsa file contain a part of the S. cerevisiae S288C genome.

In [ ]:
PRS3_locus = read("S288C_YHL011C_PRS3_flanking.fsa")

The PRS3_locus contain the coding DNA and 1000 bp up ad downstream of the orf. Downloaded from [here](https://www.yeastgenome.org/locus/S000001003/sequence).

In [ ]:
PRS3_locus

In [ ]:
from pydna.amplify import pcr

In [ ]:
PRS3_product = pcr(p1, p2, PRS3_locus)

In [ ]:
PRS3_product

In [ ]:
PRS3_product.figure()

The primers anneal perfectly to the template, so this is the PCR product we want.

# HAA1 construct

We will now do the same with the HAA1 cassette.

Vector: BHUM1737

Construction: obtained by PCR amplification of a SalI/BamHI fragment carrying HAA1 from yeast genomic DNA using appropriate primers and subsequent insertion into plasmid YEplac195.

Primers described in [Malcher et al. 2011](https://www.ncbi.nlm.nih.gov/pmc/articles/PMC3063667/) supporting information, Table S1

    HAA1hc_fw: GTC GAC CCC ATT TCC CCT TTC TTT TCC
    HAA1hc_rev: GGA TCC ATA CCT CAT CTC TGC GTG TTC G

In [ ]:
from pydna.parsers import parse_primers

In [ ]:
h1,h2 = parse_primers('''
>HAA1hc_fw
GTC GAC CCC ATT TCC CCT TTC TTT TCC
>HAA1hc_rev
GGA TCC ATA CCT CAT CTC TGC GTG TTC G
''')

In [ ]:
HAA1_locus = read("S288C_YPR008W_HAA1_flanking.fsa")

In [ ]:
HAA1_locus

In [ ]:
HAA1_product = pcr(h1, h2, HAA1_locus)

In [ ]:
HAA1_product

In [ ]:
HAA1_product.figure()

These primers are tailed, but we have no reason to include these tails (containing restriction sites). We therefore cut six bp from the beginning and six bp from the end of the sequence:

In [ ]:
h1 = h1[6:]
h2 = h2[6:]

In [ ]:
HAA1_product = pcr(h1, h2, HAA1_locus)

In [ ]:
HAA1_product.figure()

In [ ]:
HAA1_product.seq

Now we will have to design tailed primers for the HAA1_product and the PRS3_product sequences so that we can add them to pMEC1049 by gap repair. First we have to decide with which restriction enzymes we should open the pMEC1049 vector.

The restriction enzymes below are candidates for linearizing the pMEC1049 before an after the cassette.

[XhoI](http://rebase.neb.com/rebase/enz/XhoI.html) [AleI](http://rebase.neb.com/rebase/enz/AleI.html) [OliI](http://rebase.neb.com/rebase/enz/OliI.html)

These enzymes are also unique in the pYPK0 based vectors, so we can use tha same strategy to create vectors expressing only the pRS3 and/or HAA1 but without the xylose pathway if needed.

In [ ]:
from pydna.readers import read

In [ ]:
pMEC1049 = read("pMEC1049.gb")

In [ ]:
assert pMEC1049.seguid() == 'cdseguid=u60NZ3j9UWsnkLOue3T5IrZa_DY'

In [ ]:
from Bio.Restriction import XhoI, AleI, OliI

In [ ]:
pMEC1049_xho = pMEC1049.linearize(XhoI)

We design gap repair primers using the pydna assembly primers function

In [ ]:
from pydna.dseqrecord import Dseqrecord

In [ ]:
from pydna.design import assembly_fragments

In [ ]:
fragments = assembly_fragments( [Dseqrecord(pMEC1049_xho.seq.mung()), HAA1_product, pMEC1049_xho] )

In [ ]:
HAA1_product.seq

In [ ]:
Hfw = fragments[1].forward_primer
Hrv = fragments[1].reverse_primer

In [ ]:
Hfw.id = "Hfw"
Hrv.id = "Hrv"

In [ ]:
Hfw = Hfw[1:]
Hrv = Hrv[:-1]

In [ ]:
Hfw = Hfw[:50] # we limit the length to 50 bp since these are less expensive from our provider
Hrv = Hrv[:50]

In [ ]:
print( Hfw.format("tab") )

In [ ]:
print( Hrv.format("tab") )

In [ ]:
HAA1_recombination_product = pcr(Hfw, Hrv, HAA1_locus)

In [ ]:
HAA1_recombination_product

In [ ]:
HAA1_recombination_product.figure()

In [ ]:
from pydna.assembly import Assembly

In [ ]:
asm_haa1 = Assembly((pMEC1049_xho, HAA1_recombination_product))

In [ ]:
asm_haa1

In [ ]:
candidate = asm_haa1.assemble_circular()[0]

In [ ]:
candidate.figure()

In [ ]:
pMEC9001 = candidate.synced(pMEC1049)

In [ ]:
pMEC9001.stamp()

In [ ]:
assert pMEC9001.seguid() == "cdseguid=kSu-AcOo1olUXjSc3gnn0bWhezA"

In [ ]:
pMEC9001.locus="pMEC9001"

The pMEC9001 is the pMEC1049 with HAA1. The sequence can be downloaded using the link below.

In [ ]:
pMEC9001.write("pMEC9001.gb")

## PRS3

We will now make a pMEC1049 with PRS3 called pMEC9002.

In [ ]:
pMEC1049_oli = pMEC1049.linearize(OliI)

The integration site was chosen to be the uniqie OliI site.

In [ ]:
fragments2 = assembly_fragments((pMEC1049_oli, PRS3_product, pMEC1049_oli))

In [ ]:
Pfw = fragments2[1].forward_primer
Prv = fragments2[1].reverse_primer

In [ ]:
Pfw.id = "Pfw"
Prv.id = "Prv"

In [ ]:
Prv=Prv[:-2]

In [ ]:
Pfw = Pfw[:51]
Prv = Prv[:51]

In [ ]:
print( Pfw.format("tab"))
print( Prv.format("tab"))

In [ ]:
PRS3_recombination_product = pcr(Pfw, Prv, PRS3_locus)

In [ ]:
PRS3_recombination_product

The recombination was designed for OliI but AleI was used.

In [ ]:
pMEC1049_ale = pMEC1049.linearize(AleI)

In [ ]:
asm_prs3 = Assembly((pMEC1049_ale, PRS3_recombination_product))

In [ ]:
asm_prs3

In [ ]:
candidate = asm_prs3.assemble_circular()[0]

In [ ]:
candidate

In [ ]:
pMEC9002 = candidate.synced(pMEC1049)

In [ ]:
pMEC9002.locus = "pMEC9002"

In [ ]:
pMEC9002.stamp()

In [ ]:
assert pMEC9002.seguid() == "cdseguid=zl65eD-5ilNaAO6HVaGYDFHmLek"

The pMEC9002 vector is the pMEC1049 with PRS3

In [ ]:
pMEC9002.write("pMEC9002.gb")

## pMEC9003

The HAA1 and PRS3 cassettes were added to the plasmid in one step to the plasmid digested with both XhoI and AleI. Cutting with XhoI and AleI makes two fragments about 6 and 9 kb.

In [ ]:
pMEC1049_9kbp, pMEC1049_6kb = pMEC1049.cut(XhoI, AleI)

In [ ]:
pMEC1049_6kb

In [ ]:
pMEC1049_9kbp

In [ ]:
pMEC1049_6kb.locus = "pMEC1049_6kb"
pMEC1049_9kbp.locus = "pMEC1049_9kbp"

In [ ]:
asm_prs_haa = Assembly((pMEC1049_6kb, pMEC1049_9kbp, HAA1_recombination_product, PRS3_recombination_product))

In [ ]:
asm_prs_haa

In [ ]:
candidate = asm_prs_haa.assemble_circular()[0]

In [ ]:
candidate

In [ ]:
pMEC9003 = candidate.synced(pMEC1049)

In [ ]:
pMEC9003.stamp()

In [ ]:
assert pMEC9003.seguid() == "cdseguid=8bvZM2EzB5XlRewv1IEH8TvJETw"

In [ ]:
pMEC9003.locus="pMEC9003"

pMEC9003 is the pMEC1049 with both HAA1 and PRS3. The sequence can be downloaded from the link below.

In [ ]:
pMEC9003

In [ ]:
pMEC9003.write("pMEC9003.gb")